**Python ML Workflow (PyTorch / Scikit-Learn)**

**Easy**

1. What is a Pipeline?
2. Why do we use train-test split?
3. What is data leakage?
4. What is cross-validation?
5. Difference between classification and regression.

**Medium**

6. Why use model.eval()?
7. Difference between torch.no_grad() and torch.inference_mode().
8. How do BatchNorm and Dropout behave during inference?
9. How do you save and load a model?
10. Why use state_dict()?

**Hard**

11. Build a multiclass classifier using PyTorch.
12. Reduce inference latency.
13. Deploy a PyTorch model using FastAPI.
14. Optimize GPU inference.
15. Explain quantization and pruning.

## **1. What is a Pipeline**

In PyTorch and machine learning interviews, a pipeline refers to the end-to-end sequence of steps that takes raw data, processes it, trains a model, and produces predictions. It organizes the workflow so that each stage is modular and repeatable.

**Typical ML Pipeline**

```
Raw Data
    │
    ▼
Data Loading
    │
    ▼
Data Preprocessing
    │
    ▼
Feature Engineering / Augmentation
    │
    ▼
Train / Validation Split
    │
    ▼
Model Definition
    │
    ▼
Training
    │
    ▼
Validation & Hyperparameter Tuning
    │
    ▼
Testing
    │
    ▼
Model Saving
    │
    ▼
Inference (Predictions)
```

**PyTorch Pipeline Example**

In [ ]:
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# 1. Data preprocessing
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5), (0.5))
])

# 2. Load dataset
train_dataset = dataset.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

# 3. Create DataLoader
train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True
)

# 4. Define model
model = Net()

# 5. Loss and Optimizer
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters())

# Training loop
for epoch in range(5):
    for images, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

# 7. Save model
torch.save(model.state_dict(), "model.pth")

**Why Use a Pipeline?**

Modularity: Each stage (loading, preprocessing, training, evaluation) is separate and easier to maintain.
Reusability: You can reuse preprocessing or training code across projects.
Reproducibility: Running the same pipeline with the same settings gives consistent results.
Scalability: It's easier to extend the pipeline with new datasets, models, or evaluation steps.

### **2. Why do we use train-test split?**

We use a train-test split to evaluate how well a machine learning model generalizes to unseen data. The model learns from the training set, while the test set provides an unbiased estimate of its performance on new data.

```
Train on 800 images
        ↓
Learn patterns
        ↓
Predict on 200 unseen images
        ↓
Measure accuracy
```

The test accuracy is a better indicator of how the model will perform in the real world.

**Common Split Ratios**

- 80% train / 20% test (most common)
- 70% train / 30% test
- 90% train / 10% test (for large datasets)

**What about a Validation Set?**

In many projects, the data is split into three parts:

```
Dataset
   │
   ├── Train (70%)
   │      Learn model parameters
   │
   ├── Validation (15%)
   │      Tune hyperparameters and choose the best model
   │
   └── Test (15%)
          Final unbiased evaluation
```

- **Training set:** Used to update the **model's weights.**
- **Validation set:** Used to tune hyperparameters **(e.g., learning rate, batch size, number of layers)** and monitor training.
- **Test set:** Used only once at the end to **estimate final performance.**


**PyTorch Example**

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

### **3. What is data leakage?**

Data leakage occurs when information that would not be available at prediction time is used during model training. As a result, the model appears to perform very well during evaluation but fails on real-world data.

**How to Prevent Data Leakage**

Split the data into train, validation, and test sets before preprocessing.
Fit preprocessing steps (e.g., scaling, encoding, imputation) only on the training set, then apply them to validation and test sets.
Avoid using future information or target-derived features.
Ensure there are no duplicate or overlapping samples between datasets.
When working with time-series data, train on past data and test on future data.

## **4. What is cross-validation?**

A single train-test split can give misleading results because the performance depends on which samples ended up in the training and test sets.

Cross-validation reduces this bias by evaluating the model multiple times on different splits.

**K-Fold Cross-Validation**

Suppose you have **1,000 samples** and choose **5-fold cross-validation.**

```
Fold 1 | Fold 2 | Fold 3 | Fold 4 | Fold 5
```

The model is trained and tested 5 times:

```
Iteration 1:
Train: Fold 2 + 3 + 4 + 5
Test : Fold 1

Iteration 2:
Train: Fold 1 + 3 + 4 + 5
Test : Fold 2

Iteration 3:
Train: Fold 1 + 2 + 4 + 5
Test : Fold 3

Iteration 4:
Train: Fold 1 + 2 + 3 + 5
Test : Fold 4

Iteration 5:
Train: Fold 1 + 2 + 3 + 4
Test : Fold 5
```

The final performance is the average of all 5 test scores.

**Example**
Accuracy obtained in each fold:

```
Fold 1 = 91%
Fold 2 = 89%
Fold 3 = 92%
Fold 4 = 90%
Fold 5 = 88%
```

Average accuracy: ``` (91 + 89 + 92 + 90 + 88) / 5 = 90%```

This average is usually a more reliable estimate than a single train-test-split.

**Advantages**
- Uses the dataset more efficiently.
- Reduces the effect of a lucky or unlucky train-test-split
- Provides a more stable estimate of model performance.
- Helps compare different models and tune hyperparameter.

**Disadvantages**
- More computationally expensive because the model is trained multiple times.
- Can be slow for large datasets or deep learning models.

**Python Example**

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier()

scores = cross_val_score(
    model,
    x,
    y, 
    cv=5
)

print(score)
print(score.mean())

**Cross-Validation in Deep Learning**

For traditional machine learning, 5-fold or 10-fold cross-validation is common.

For deep learning (e.g., PyTorch), cross-validation is less common because training neural networks is computationally expensive. Instead, practitioners often use:

- Train/validation/test split
- Early stopping
- Hyperparameter tuning using the validation set

However, cross-validation is still useful when:

- The dataset is small.
- You need a robust estimate of performance (e.g., in medical or scientific applications).

### **5. Difference between classification and Regression**

| Feature                | Classification                                                          | Regression                                                       |
| ---------------------- | ----------------------------------------------------------------------- | ---------------------------------------------------------------- |
| **Goal**               | Predict a category or class                                             | Predict a continuous numerical value                             |
| **Output**             | Discrete labels                                                         | Continuous values                                                |
| **Examples**           | Spam/Not Spam, Cat/Dog, Fraud/Not Fraud                                 | House price, Temperature, Salary                                 |
| **Target Variable**    | Categorical                                                             | Numeric                                                          |
| **Common Algorithms**  | Logistic Regression, Decision Tree, Random Forest, SVM, Neural Networks | Linear Regression, Random Forest Regressor, SVR, Neural Networks |
| **Evaluation Metrics** | Accuracy, Precision, Recall, F1-score, ROC-AUC                          | MAE, MSE, RMSE, R²                                               |


## **Medium**

## 6. What does model.eval() do?

`model.eval()` switches the model from training mode to evaluation (inference) mode.

It changes the behavior of certain layers, mainly:

- Dropout
- Batch Normalization

**Why is it needed?**

During training, some layers behave differently than they should during inference.

**1. Dropout**

During training:

```
model.train()
```
Dropout randomly turns off neurons to reduce overfitting.

**Example:**
```
Input neurons:
1 1 1 1 1

After Dropout:
1 0 1 0 1
```
During testing, you **do not** want random neurons to be disabled.
```
model.eval()
```
Dropout is disabled, so all neurons are used.


**2. Batch Normalization**

During training:

BatchNorm computes the `mean` and `variance` from the current mini-batch.

During evaluation:

BatchNorm uses the `running mean` and `running variance` learned during training instead of the current batch statistics.

This makes predictions stable and consistent.

**Example**

In [ ]:
model.eval()

with torch.no_grad():
    outputs = model(images)

Here:

- `model.eval()` changes the behavior of Dropout and BatchNorm.
- `torch.no_grad()` disables gradient computation, reducing memory usage and speeding up inference.

These serve different purposes and are commonly used together during evaluation.

### **7. Difference between torch.no_grad() and torch.inference_mode()**

`torch.no_grad()` disables gradient computation, while `torch.inference_mode()` disables gradients and additional autograd bookkeeping, making inference faster and more memory-efficient. `torch.inference_mode()` is recommended for pure inference when you don't need to modify tensors that require gradients.


`torch.no_grad():` **Don't compute or store gradients.**

**Example:**
```
model.eval()

with torch.no_grad():
    outputs = model(images)
```

**Benefits**
- Saves memory
- Speeds up inference
- Prevents accidental gradient computation

`torch.inference_mode():` **Introduced as a faster alternative for inference.**

```
model.eval()

with torch.inference_mode():
    outputs = model(images)
```

It does everything `torch.no_grad()` does plus:

- Disables gradient computation
- Disables extra autograd tracking (such as version counter updates)
- Reduces overhead
- Can improve inference performance


| Feature                             | `torch.no_grad()` | `torch.inference_mode()` |
| ----------------------------------- | ----------------- | ------------------------ |
| Disables gradients                  | ✅                 | ✅                        |
| Saves memory                        | ✅                 | ✅                        |
| Faster inference                    | ✅                 | ✅ (usually faster)       |
| Disables extra autograd bookkeeping | ❌                 | ✅                        |
| Best for production inference       | Good              | **Recommended**          |


**When to Use Which?**
Use `torch.no_grad()` when:
- Evaluating a model during training
- Running validation
- You may still need some flexibility with tensors and autograd outside the context

```
model.eval()

with torch.no_grad():
    val_loss = ...
```

Use `torch.inference_mode()` when:
- Deploying a model
- Serving predictions through an API
- Running production inference
- Benchmarking inference speed

**Example:**
```
model.eval()

with torch.inference_mode():
    prediction = model(image)
```

### **Why is torch.inference_mode() faster?**

Because it not only disables gradient computation but also skips additional autograd bookkeeping (such as version counter updates), reducing overhead during inference.

### **8. How do BatchNorm and Dropout behave during inference?**

**1. Dropout During Inference**
Training Mode (model.train())

Dropout randomly sets some neuron activations to zero.

**Example:**

```
Input:
[2, 5, 3, 1]

After Dropout (p=0.5):
[2, 0, 3, 0]
```
This helps prevent overfitting by ensuring the model doesn't rely too heavily on any one neuron.

Inference Mode `(model.eval())`

Dropout is disabled.

```
Input:
[2, 5, 3, 1]

Output:
[2, 5, 3, 1]
```
Every neuron is used, giving deterministic predictions for the same input.

**2. Batch Normalization During Inference**

**Training Mode**

For each mini-batch, BatchNorm:

- Computes the batch mean
- Computes the batch variance
- Normalizes the activations
- Updates the running mean and running variance

**Summary**

|Layer|	Training (model.train())|	Inference (model.eval())|
|-----|-------------------------|---------------------------|
|Dropout|	Randomly drops neurons|	Disabled; all neurons are active|
|BatchNorm|	Uses current batch mean and variance; |updates running statistics	Uses stored running mean and variance



**Why is model.eval() Important?**
```
model.eval()
```
- Dropout continues randomly dropping neurons.
- BatchNorm uses the current batch's statistics instead of the learned running statistics.

This can lead to inconsistent predicition and inaccurate evaluation metrics.

### **9. How do you save and load a model?**

**1. Save the Model**
The recommended approach is to save the model's parameters:

```
torch.save(model.state_dict(), "model.pth")
```
This creates a file (model.pth) containing the model weights.

**2. Load the Model**
First, recreate the model architecture:

```
model = Net()  # Same architecture used during training
```
Then load the saved weights:

```
model.load_state_dict(torch.load("model.pth"))
model.eval()
```

`model.eval()` ensures that layers like Dropout and BatchNorm behave correctly during inference.

**3. Why Save `state_dict()` Instead of the Entire Model?**

`state_dict()` contains only the model's learned parameters.

Advantages:

- Smaller file size
- More portable across projects
- Less dependent on the exact Python class implementation
- Recommended by the PyTorch documentation

**4. Saving the Entire Model (Possible but Not Recommended)**

```
torch.save(model, "model.pth")
```

Loading:
```
model = torch.load("model.pth")
```

**Why is this discouraged?**

- It depends on the exact model class definition being available.
- Changes to the code can make old saved models difficult or impossible to load.
- It is less flexible than saving the state_dict.

**5. Saving a Checkpoint**
During training, it's common to save more than just the model:

```
torch.save({
    "epoch": epoch,
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "loss": loss,
}, "checkpoint.pth")
```

Load it later:

```
checkpoint = torch.load("checkpoint.pth")

model.load_state_dict(checkpoint["model_state_dict"])
optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

epoch = checkpoint["epoch"]
loss = checkpoint["loss"]
```
This allows you to resume training from where it stopped.

**6. Save vs. Checkpoint**

| Save Model           | Save Checkpoint                                                       |
| -------------------- | --------------------------------------------------------------------- |
| Stores model weights | Stores model weights + optimizer state + epoch + other training state |
| Used for inference   | Used to resume training                                               |
| Smaller file         | Larger file                                                           |


### 10. Why use `state_dict()`?

```
torch.save(model.state_dict(), "model.pth")
```

**Advantage:**
- Saves only the parameters.
- Smaller file size.
- More portable
- Easy to load into the same architecture
- Recommended by PyTorch.

## **Hard**

### **11. Build a multiclass classifier using PyTorch**

**Assume**
- Input feature: 20
- Number of classess: 5
- Loss: CrossEntropyLoss
- Optimizer: Adam

In [ ]:
# Step 1: Import libraries
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

In [ ]:
# Step 2: Create Sample Dataset

# 1000 samples, 20 features
X = torch.randn(1000, 20)

# Labels: 0,1,2,3,4
y = torch.randint(0, 5, (1000,))

dataset = TensorDataset(X, y)
loader = DataLoader(dataset, batch_size=32, shuffle=True)

In [ ]:
# Step 3: Define the model
class MultiClassClassifier(nn.Module):

    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(20, 64),
            nn.ReLU(),

            nn.Linear(64, 32),
            nn.ReLU(),

            nn.Linear(32, 5) # 5 output classes
        )

    def forward(self, x):
        return self.network(x)

In [ ]:
# Initialize Model
model = MutiClassClassifier()

In [ ]:
# Step 5: Loss and Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [ ]:
# Step 6: Training Loop

epochs = 10

for epoch in range(epochs):
    model.train()

    running_loss =0

    for inputs, labels in loader:
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    print(f"Epoch {epoch+1}: Loss ={running_loss:.4f}")

In [ ]:
# Step 7: Evaluation
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for inputs, labels in loader:
        outputs = model(inputs)
        predicitions = torch.argmax(outputs, dim=1)

        correct+=(predictions == labels).sum().item()
        total+=labels.size(0)

accuracy = correct / total

print(f"Accuracy: {accuracy:.2f%}")

In [ ]:
# Step 8: Save the model
torch.save(model.state_dict(), "multiclass_model.pth")

In [ ]:
# Step 9: Load the Model
model = MultiClassClassifier()
model.load_state_dict(torch.load("multiclass_model.pth"))
model.eval()

### **12. Reduce inference latency**

**1. Use model.eval()**
```
model.eval()
```
**why?**
- Disable Dropout
- Uses BatchNorm running Statistics
- Ensures correct inference behavior
  
**2. Disable Gradient Computation**

```
with torch.inference_mode():
    output = model(x)
```
Or
```
with torch.no_grad():
    output = model(x)
```
**Benefits:**

- Less memory
- Faster execution
- No autograd overhead

**3. Use GPU (if available)**
```
device = torch.device("cuda")

model.to(device)
x = x.to(device)
```
GPU acceleration can significantly reduce inference time for large models or batches.

**4. Mixed Precision Inference**
Use FP16 on supported GPUs.
```
with torch.inference_mode():
    with torch.autocast(device_type="cuda", dtype=torch.float16):
        output = model(x)
```
**Benefits:**

- Faster inference
- Lower GPU memory usage
  
**5. Quantization**
Convert weights from FP32 to INT8 where appropriate.

**Benefits:**

- Smaller model
- Faster CPU inference
- Reduced memory usage

**6. TorchScript or torch.compile()**
PyTorch provides ways to optimize execution.

Example (PyTorch 2.x):
```
model = torch.compile(model)
```
**Benefits:**

- Fuses operations
- Optimizes execution graph
- Can reduce inference latency depending on the model and hardware

**7. Smaller Model**
Instead of: **ResNet152**

use: ResNet18, MobileNetV3, EfficientNet-Lite

**8. Pruning**
Remove less important weights or channels.

**Benefits:**
- Fewer computations
- Smaller model
  
**9. Optimize Batch Size**
Latency depends on workload.

- Batch size = 1: Lowest latency for real-time requests.
- Larger batches: Higher throughput but increased latency per request.

**10. Optimize Data Loading**
Avoid unnecessary preprocessing during inference.

**Examples:**

- Resize images efficiently
- Cache repeated computations
- Use multiple workers if preprocessing is significant

**11. Export to an Optimized Runtime**
Examples include:

- TorchScript
- ONNX Runtime
- TensorRT (NVIDIA GPUs)
These runtimes can perform graph optimizations and operator fusion.

**12. Reduce Input Resolution**
Example: ``` 224x224``` -> ```160x160```
Smaller inputs require fewer computations, often reducing latency.


| Technique                          | Benefit                                             |
| ---------------------------------- | --------------------------------------------------- |
| `model.eval()`                     | Correct inference behavior                          |
| `torch.inference_mode()`           | Disables gradients and reduces autograd overhead    |
| GPU inference                      | Faster computation for suitable workloads           |
| Mixed precision (FP16)             | Faster execution and lower memory on supported GPUs |
| Quantization                       | Faster CPU inference and smaller models             |
| `torch.compile()`                  | Graph-level optimizations                           |
| Smaller architecture               | Fewer parameters and operations                     |
| Pruning                            | Reduced computation                                 |
| Optimized batch size               | Balance latency and throughput                      |
| Optimized runtimes (ONNX/TensorRT) | Hardware-specific acceleration                      |
| Smaller input size                 | Less computation                                    |


## **13. Deploy a PyTorch model using FastAPI**

**Project Structure**
```
project/
│
├── app.py
├── model.py
├── model.pth
└── requirements.txt
```

In [ ]:
# Step 1: Define the Model (model.py)

import torch.nn as nn

class Net(nn.Module):
    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(20, 64),
            nn.ReLU(),
            nn.Linear(64, 5)
        )

    def forward(self, x):
        return self.network(x)

In [ ]:
# Step 2: Load the Model (app.py)
import torch
from fastapi import FastAPI
from pydantic import BaseModel

from model import Net

app = FastAPI()

model = Net()
model.load_state_dict(torch.load('model.pth', map_location='cpu'))
model.eval()

The model is loaded once when the application starts, avoiding the overhead of reloading it for every request.

In [ ]:
# Step 3: Define the Input Schema

class InputData(BaseModel):
    feature: list[float]

**Example request:**

```
{
  "features": [
    0.2,
    1.5,
    0.8,
    ...
  ]
}
```

In [ ]:
# Step 4: Create the Prediction Endpoint

@app.post("/predict")
def predict(data: InputData):

    x = torch.tensor(data.features).float().unsqueeze(0)

    with torch.inference_mode():
        output = model(x)

        prediction = torch.argmax(output, dim=1).item()
    return {
        "prediction": prediction
    }

In [ ]:
# Step 5: Run the Server

uvicorn app:app --reload

Server: http://127.0.0.1:8000

Swagger documentation: http://127.0.0.1:8000/docs

FastAPI automatically generates interactive API documentation.

**Step 6: Test the API**

Using `curl:`
```
curl -X POST "http://127.0.0.1:8000/predict" \
-H "Content-Type: application/json" \
-d '{
    "features":[
        0.2,0.3,0.5,0.6,0.8,
        0.1,0.2,0.7,0.9,0.4,
        0.3,0.8,0.2,0.5,0.6,
        0.9,0.2,0.1,0.4,0.7
    ]
}'

```

**Example response:**

```
{
    "prediction": 3
}
```

## **15. Explain quantization and pruning.**

**What is Quantization?**
Quantization reduces the precision of the model's **weights and/or activations**.

**example:**
```
FP32 (32-bit floating point)
        ↓
FP16 (16-bit floating point)
        ↓
INT8 (8-bit integer)
```
Instead of storing weights as 32-bit floating-point numbers, they are stored with lower precision.

**Original weights:**
```
0.12345678
-0.78965432
0.56473829
```

**After INT8 quantization:**
```
12
-79
56
```
The runtime keeps track of a scale (and often a zero-point) to map these integers back to approximate floating-point values during computation.

**Benefits**
- Faster inference (especially on CPUs and supported accelerators)
- Lower memory usage
- Lower power consumption

**Trade-off**
- Slight loss in accuracy may occur, depending on the model and quantization method.

**Types of Quantization**
1. Dynamic Quantization
    - Quantizes weights.
    - Activations are quantized dynamically during inference.
    - Simple to apply.

2. Static Quantization
    - Quantizes both weights and activations.
    - Requires calibration on representative data.
    - Typically offers better performance.

3. Quantization-Aware Training (QAT)
    - Simulates quantization during training.
    - Usually achieves higher accuracy than post-training quantization.

**2. Pruning**

What is Pruning?

Pruning removes less important weights or neurons from a neural network.

Original weights:
```
0.45
0.001
-0.0003
0.72
```

**After pruning:**
```
0.45
0
0
0.72
```
Very small weights are set to zero because they contribute little to the final prediction.

**Benefits**
- Smaller effective model
- Fewer computations (especially with structured pruning and supporting runtimes)
- Lower memory usage

**Types of Pruning**
    - **Unstructured Pruning** : Removes individual weights.
```
Original

0.4 0.2 0.1
0.5 0.8 0.3

↓

0.4 0   0
0.5 0.8 0
```

Produces sparse matrices, but standard hardware may not always see large speedups without sparse kernel support.

**Structured Pruning**

Removes entire structures such as:

- Filters
- Channels
- Attention heads
- Neurons

**Example**
```
64 Filters

↓

48 Filters
```
This changes the model architecture and often leads to more practical inference speedups on standard hardware.